In [23]:

import pandas as pd
import geopandas as gpd

pluto = pd.read_csv("../data/processed/manhattan_pluto_26v2_cleaned.csv.gz")
nta = gpd.read_file("../data/raw/nynta2020.shp")

manhattan_nta = nta[nta["BoroCode"] == 1]

pluto_gdf = gpd.GeoDataFrame(
    pluto,
    geometry=gpd.points_from_xy(pluto["longitude"], pluto["latitude"]),
    crs="EPSG:4326"
).to_crs(manhattan_nta.crs)

pluto_with_nta = gpd.sjoin(
    pluto_gdf,
    manhattan_nta[["NTA2020", "NTAName", "geometry"]],
    how="left",
    predicate="within"
)

pluto_by_nta = (
    pluto_with_nta.groupby(["NTA2020", "NTAName"], as_index=False)
    .agg(
        total_residential_units=("unitsres", "sum"),
        tax_lot_count=("bbl", "count")
    )
)

pluto_by_nta.to_csv(
    "../data/processed/manhattan_pluto_units_by_nta.csv",
    index=False
)

pluto_by_nta


,NTA2020,NTAName,total_residential_units,tax_lot_count
0,MN0101,Financial District-Battery Park City,36436.0,554
1,MN0102,Tribeca-Civic Center,12249.0,866
2,MN0191,The Battery-Governors Island-Ellis Island-Libe...,0.0,9
3,MN0201,SoHo-Little Italy-Hudson Square,15323.0,1577
4,MN0202,Greenwich Village,21491.0,1081
5,MN0203,West Village,24535.0,2014
6,MN0301,Chinatown-Two Bridges,17754.0,1021
7,MN0302,Lower East Side,25685.0,921
8,MN0303,East Village,40193.0,2278
9,MN0401,Chelsea-Hudson Yards,46130.0,1923


notes: 
- Spatially joined Manhattan PLUTO properties to 2020 NTA boundaries
- Aggregated residential units and tax-lot counts by NTA
- Matched 41,688 properties across all 38 Manhattan NTAs, totaling 955,263 residential units
- Excluded 753 unmatched properties containing 3,693 residential units (0.39% of total)